# **Проект: Обучение визуально-языковой модели**


---
### **Описание проекта:**

Дообучение визуально-языковой модели **Qwen2-VL-2B-Instruct** на русскоязычном датасете **GQA-ru** с использованием метода QLoRA.

**Цель:** получить более высокие метрики на бенчмарках GQA-ru и MMBench-ru.

*Проект выполнен в условиях ресурсных ограничений Google Colab T4.*

**Этапы выполнения:**


1. Установка зависимостей
2. Загрузка и оценка базовой модели Qwen/Qwen2-VL-2B-Instruct на бенчмарках GQA-ru и MMBench-ru
3. Загрузка и форматирование датасета GQA-ru
4. Настройка параметров обучения
5. Обучение базовой модели
6. Оценка обученной модели на бенчмарках GQA-ru и MMBench-ru
7. Сравнение результатов






## **1) Установка библиотек и вход в Hagging Face**

In [ ]:
!pip install -U -q git+https://github.com/huggingface/trl.git bitsandbytes peft qwen-vl-utils
!git clone https://github.com/EvolvingLMMs-Lab/lmms-eval.git
%cd lmms-eval
!pip install -q -e .
%cd ..
!pip install -q decord

from huggingface_hub import login
login()

## **2) Оценка базовой модели Qwen/Qwen2-VL-2B-Instruct**

Для оценки модели используется фреймворк lmms-eval.
Взят `--limit 200` от каждого бенчмарка GQA-ru и MMBench-ru в целях уменьшения времени выполнения оценивания.

In [ ]:
!python -m lmms_eval \
  --model qwen2_vl \
  --model_args pretrained="Qwen/Qwen2-VL-2B-Instruct",device_map="auto" \
  --tasks gqa-ru \
  --limit 200 \
  --batch_size 2 \
  --output_path ./results/qwen2_vl_2b/gqa_ru \
  --log_samples

In [ ]:
!python -m lmms_eval \
  --model qwen2_vl \
  --model_args pretrained="Qwen/Qwen2-VL-2B-Instruct",device_map="auto" \
  --tasks mmbench_ru_dev \
  --limit 200 \
  --batch_size 2 \
  --output_path ./results/qwen2_vl_2b/mmbench_ru \
  --log_samples

## **3) Загрузка и форматирование датасета GQA-ru**

**Структура датасета GQA-ru:**

* train сплит - сабсеты train_balanced_instructions и train_balanced_images
* testdev сплит - сабсеты testdev_balanced_instructions и testdev_balanced_images

Train сплит содержит 27 519 изображений с 40 000 вопросами к ним,  testdev сплит - 398 изображений с 12 216 вопросами к ним.



> Для обучения модели возьмем 2 000 случайных запросов из train сплита, которые разделим на обучающие и валидационные данные (80% / 20%).

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
from PIL import Image
import io


dataset_id = "deepvk/GQA-ru"

train_instr_dataset = load_dataset(dataset_id, "train_balanced_instructions", split='train').to_pandas()
train_img_dataset  = load_dataset(dataset_id, "train_balanced_images", split='train').to_pandas()
# Объединение датасетов
train_img_dataset = train_img_dataset.rename(columns={'id': 'imageId'})
train_dataset = pd.merge(train_instr_dataset, train_img_dataset, on='imageId')


def format_data(sample):
  question = sample["question"]
  prompt = f"{question} Ответь одним словом."
  return {
    "image": Image.open(io.BytesIO(sample['image']['bytes'])),
    "messages": [
      {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": prompt}],
      },
      {
        "role": "assistant",
        "content": [{"type": "text","text": sample["answer"]}],
      }
    ]
  }


# Для обучения и валидации используем 1000 запросов
train_dataset = train_dataset.sample(frac=1, random_state=42).reset_index(drop=True) # Перемешиваем данные
train_dataset_short = train_dataset[:2000].to_dict('records')
train_dataset_format = [format_data(sample) for sample in train_dataset_short]

# Разделяем на train и eval (80% / 20%)
split_idx = int(len(train_dataset_format) * 0.8)
train_ds = train_dataset_format[:split_idx]
eval_ds = train_dataset_format[split_idx:]

print(f"Train set: {len(train_ds)}")
print(f"Eval set: {len(eval_ds)}")

train_dataset = Dataset.from_list(train_ds)
eval_dataset = Dataset.from_list(eval_ds)

**Очистка ресурсов перед началом обучения**

In [ ]:
import torch
import gc
import time


def clear_memory():
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    heavy_vars = [
        'train_instr_dataset', 'train_img_dataset',
        'train_dataset_short', 'train_dataset_format',
        'train_ds', 'eval_ds'
    ]

    for var_name in heavy_vars:
        if var_name in globals():
            del globals()[var_name]

    # Сборка мусора и очистка памяти CUDA
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory()

## **4) Загрузка модели с квантованием, настройка LoRA и обучения**

Уменьшаем вес модели с помощью 4-битного квантования (QLoRA) и загружаем ее. Настраиваем LoRA (PEFT).

Для адаптации предварительно обученной модели к нашей задаче мы будем использовать Supervised Fine-Tuning (SFT), зададим параметры через SFTConfig.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig


model_id = "Qwen/Qwen2-VL-2B-Instruct"

# BitsAndBytesConfig int-4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)
processor = Qwen2VLProcessor.from_pretrained(model_id)


# Настройка LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)


# Параметры обучения
training_args = SFTConfig(
    output_dir="qwen2-vl-2b-gqa-ru-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    logging_steps=10,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none",
    push_to_hub=True, # модель будет загружена на ваш Hugging Face
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True}
)

## **5) Обучение базовой модели**

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=processor,
)

trainer.train()
trainer.save_model(training_args.output_dir)

## **6) Оценка обученной модели**

Для этого очистим память, перезагрузим базовую модель и присоединим к ней обученный адаптер.

In [ ]:
from peft import PeftModel

clear_memory()

model_id = "Qwen/Qwen2-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)
processor = Qwen2VLProcessor.from_pretrained(model_id)

adapter_path = "HopeitCanbeChanged/qwen2-vl-2b-gqa-ru-finetuned" # загружаем адаптер с Hugging Face
merged_model_path = "./qwen2-vl-2b-merged"
peft_model = PeftModel.from_pretrained(model, adapter_path)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(merged_model_path)
processor = Qwen2VLProcessor.from_pretrained(model_id)
processor.save_pretrained(merged_model_path)

del model, peft_model, merged_model, processor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!python -m lmms_eval \
  --model qwen2_vl \
  --model_args pretrained="qwen2-vl-2b-merged",device_map="auto" \
  --tasks gqa-ru \
  --limit 200 \
  --batch_size 2 \
  --output_path ./results/qwen2_vl_2b_finetuned/gqa_ru \
  --log_samples

In [ ]:
!python -m lmms_eval \
  --model qwen2_vl \
  --model_args pretrained="qwen2-vl-2b-merged",device_map="auto" \
  --tasks mmbench_ru_dev \
  --limit 200 \
  --batch_size 2 \
  --output_path ./results/qwen2_vl_2b_finetuned/mmbench_ru \
  --log_samples

## **7) Результаты**

In [11]:
import glob
import json


paths = {
    "baseline GQA-ru": "./results/qwen2_vl_2b/gqa_ru/Qwen__Qwen2-VL-2B-Instruct/*.json",
    "baseline MMBench-ru": "./results/qwen2_vl_2b/mmbench_ru/Qwen__Qwen2-VL-2B-Instruct/*.json",
    "finetuned GQA-ru": "./results/qwen2_vl_2b_finetuned/gqa_ru/qwen2-vl-2b-merged/*.json",
    "finetuned MMBench-ru":"./results/qwen2_vl_2b_finetuned/mmbench_ru/qwen2-vl-2b-merged/*.json"
}
metrics = {}

for name, pattern in paths.items():
    file = glob.glob(pattern)[0]
    if file:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        results = data["results"]
        value = 0
        if results.get("gqa-ru"):
          value = results["gqa-ru"].get("exact_match,none")
          value = value * 100
        elif results.get("mmbench_ru_dev"):
          value = results["mmbench_ru_dev"].get("gpt_eval_score,none")

        if isinstance(value, (int, float)):
            metrics[name] = f"{value:.2f}%"
        else:
            metrics[name] = "N/A"
    else:
        metrics[name] = "Файл не найден"


df = pd.DataFrame({
    "Модель": ["Baseline", "Finetuned"],
    "GQA-ru (Exact Match)": [metrics["baseline GQA-ru"], metrics["finetuned GQA-ru"]],
    "MMBench-ru (GPT Eval Score)": [metrics["baseline MMBench-ru"], metrics["finetuned MMBench-ru"]]
})

print(df.to_string(index=False))
df.to_csv("results.csv", index=False, encoding="utf-8-sig")

   Модель GQA-ru (Exact Match) MMBench-ru (GPT Eval Score)
 Baseline               27.00%                      56.00%
Finetuned               31.00%                      56.50%
